In [2]:
import pandas as pd

In [3]:
temp = pd.read_csv("GLB.Ts+dSST.csv")
co2 = pd.read_csv("annual-co2-emissions-per-country.csv")
aqi = pd.read_csv("global_air_quality_dataset.csv")

print("Temperature shape:", temp.shape)
print("CO2 shape:", co2.shape)
print("AQI shape:", aqi.shape)

Temperature shape: (148, 1)
CO2 shape: (29384, 4)
AQI shape: (3660, 13)


In [4]:
temp = pd.read_csv("GLB.Ts+dSST.csv", skiprows=1)

temp.head()

,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
0,1880,-0.19,-.25,-.10,-.17,-.11,-.22,-.19,-.11,-.15,-.24,-.23,-.18,-.18,***,***,-.13,-.17,-.21
1,1881,-0.21,-.15,.02,.04,.05,-.20,-.01,-.04,-.16,-.22,-.19,-.08,-.10,-.10,-.18,.04,-.08,-.19
2,1882,0.15,.13,.04,-.18,-.15,-.24,-.17,-.08,-.15,-.24,-.17,-.37,-.12,-.09,.07,-.10,-.16,-.19
3,1883,-0.30,-.37,-.13,-.19,-.18,-.07,-.08,-.14,-.22,-.11,-.25,-.12,-.18,-.20,-.35,-.17,-.10,-.19
4,1884,-0.14,-.09,-.37,-.40,-.34,-.35,-.31,-.28,-.27,-.25,-.34,-.31,-.29,-.27,-.11,-.37,-.31,-.29


In [5]:
# Keep only Year and Annual mean column
temp = temp[['Year', 'J-D']]

# Rename columns
temp.columns = ['year', 'avg_temperature']

# Convert year to numeric
temp['year'] = pd.to_numeric(temp['year'], errors='coerce')

# Remove invalid rows
temp = temp.dropna()

# Convert year to integer
temp['year'] = temp['year'].astype(int)

# Keep recent years only (2015 onwards)
temp = temp[temp['year'] >= 2015]

temp.head()

,year,avg_temperature
135,2015,.90
136,2016,1.01
137,2017,.92
138,2018,.85
139,2019,.98


In [6]:
co2.head()

,Entity,Code,Year,Annual CO₂ emissions
0,Afghanistan,AFG,1949,14656.0
1,Afghanistan,AFG,1950,84272.0
2,Afghanistan,AFG,1951,91600.0
3,Afghanistan,AFG,1952,91600.0
4,Afghanistan,AFG,1953,106256.0


In [7]:
# Keep required columns
co2 = co2[['Entity', 'Year', 'Annual CO₂ emissions']]

# Rename columns
co2.columns = ['country', 'year', 'co2']

# Convert year to numeric
co2['year'] = pd.to_numeric(co2['year'], errors='coerce')

# Remove invalid rows
co2 = co2.dropna()

# Convert year to int
co2['year'] = co2['year'].astype(int)

# Keep recent years only
co2 = co2[co2['year'] >= 2015]

co2.head()

,country,year,co2
66,Afghanistan,2015,9384400.0
67,Afghanistan,2016,8605932.0
68,Afghanistan,2017,9311054.0
69,Afghanistan,2018,10191504.0
70,Afghanistan,2019,10400110.0


In [8]:
aqi.head()

,Date,City,Country,AQI,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (ppb),SO2 (ppb),CO (ppm),O3 (ppb),Temperature (°C),Humidity (%),Wind Speed (m/s)
0,2024-01-01,New York,USA,38,120.0,182.9,24.3,26.0,9.10,153.3,18.6,40,13.2
1,2024-01-01,Los Angeles,USA,280,38.4,46.9,41.8,34.7,3.78,190.7,-2.2,59,9.5
2,2024-01-01,London,UK,117,168.1,34.3,81.5,8.2,3.67,105.4,36.3,62,3.4
3,2024-01-01,Beijing,China,197,96.8,35.4,18.5,39.4,9.51,92.8,29.9,32,1.8
4,2024-01-01,Delhi,India,187,76.2,226.8,46.9,17.2,1.02,68.4,9.9,55,3.3


In [10]:
# Convert Date to datetime
aqi['Date'] = pd.to_datetime(aqi['Date'])

# Extract year
aqi['year'] = aqi['Date'].dt.year

# Keep recent years
aqi = aqi[aqi['year'] >= 2015]

# Group yearly average AQI per country
aqi_yearly = aqi.groupby(['Country', 'year'])['AQI'].mean().reset_index()

# Rename columns to match other datasets
aqi_yearly.columns = ['country', 'year', 'avg_aqi']

aqi_yearly.head()

,country,year,avg_aqi
0,Australia,2024,159.620219
1,Brazil,2024,168.765027
2,China,2024,162.953552
3,Egypt,2024,166.062842
4,France,2024,162.161202


In [11]:
aqi_yearly.shape

(9, 3)

In [12]:
merged = pd.merge(co2, aqi_yearly, on=['country', 'year'], how='inner')

merged.head()

,country,year,co2,avg_aqi
0,Australia,2024,3.867324e+08,159.620219
1,Brazil,2024,4.830116e+08,168.765027
2,China,2024,1.228904e+10,162.953552
3,Egypt,2024,2.583679e+08,166.062842
4,France,2024,2.641556e+08,162.161202


In [13]:
merged.shape

(7, 4)

In [14]:
final = pd.merge(merged, temp, on='year', how='left')

final.head()

,country,year,co2,avg_aqi,avg_temperature
0,Australia,2024,3.867324e+08,159.620219,1.28
1,Brazil,2024,4.830116e+08,168.765027,1.28
2,China,2024,1.228904e+10,162.953552,1.28
3,Egypt,2024,2.583679e+08,166.062842,1.28
4,France,2024,2.641556e+08,162.161202,1.28


In [15]:
final.shape

(7, 5)

In [17]:
final.dtypes

country             object
year                 int64
co2                float64
avg_aqi            float64
avg_temperature     object
dtype: object

In [19]:
# Replace *** with NaN
final['avg_temperature'] = final['avg_temperature'].replace('***', None)

# Convert to numeric
final['avg_temperature'] = pd.to_numeric(final['avg_temperature'], errors='coerce')

# Remove any invalid rows
final = final.dropna()

final.dtypes

country             object
year                 int64
co2                float64
avg_aqi            float64
avg_temperature    float64
dtype: object

In [20]:
final['climate_risk_index'] = (
    (final['avg_temperature'] * 0.4) +
    (final['co2'] * 0.3) +
    (final['avg_aqi'] * 0.3)
)

final.head()

,country,year,co2,avg_aqi,avg_temperature,climate_risk_index
0,Australia,2024,3.867324e+08,159.620219,1.28,1.160198e+08
1,Brazil,2024,4.830116e+08,168.765027,1.28,1.449035e+08
2,China,2024,1.228904e+10,162.953552,1.28,3.686711e+09
3,Egypt,2024,2.583679e+08,166.062842,1.28,7.751041e+07
4,France,2024,2.641556e+08,162.161202,1.28,7.924674e+07


In [21]:
final.to_csv("final_climate_dataset.csv", index=False)
final.shape

(7, 6)

In [22]:
from sklearn.preprocessing import MinMaxScaler

In [23]:
scaler = MinMaxScaler()

final[['co2_norm','aqi_norm','temp_norm']] = scaler.fit_transform(
    final[['co2','avg_aqi','avg_temperature']]
)

In [24]:
final['climate_risk_index'] = (
    0.4 * final['temp_norm'] +
    0.3 * final['co2_norm'] +
    0.3 * final['aqi_norm']
)

In [25]:
final[['country','climate_risk_index']].head()

,country,climate_risk_index
0,Australia,0.003201
1,Brazil,0.255876
2,China,0.391226
3,Egypt,0.176321
4,France,0.069686


In [26]:
final.to_csv("final_climate_dataset.csv", index=False)